# Lecture 2 — Gradient descent and line search

**Week 2 · Day 2 · 45 min**

> **Headline.** The simplest optimizer, done right — and the one loop every later
> optimizer plugs into.

Yesterday we built the object to minimize. Today we descend it. The loop you write this
afternoon is written **once**: Newton on day 4 is not a new loop, it is a new
`IDirectionRule`; Armijo is not a new loop, it is a new `ILineSearch`. If on day 4 you find
yourself editing `DescentOptimizer`, something has been put in the wrong class.

**By the end of this lecture you can:**

1. state why a small enough step *must* decrease the objective, and how small is small enough;
2. derive the convergence rate $\frac{\kappa-1}{\kappa+1}$ and explain the zigzag;
3. implement Armijo backtracking and say what $c_1$ buys you;
4. explain why momentum improves $\kappa$ to $\sqrt{\kappa}$.

**You implement this afternoon:** `DescentOptimizer`, `SteepestDescent`, `HeavyBall`,
`FixedStep`, `Armijo`, the stopping criteria, and `History`.

### Pacing

Target **40 min** of core material, hard cap **45 min**. Sections marked
*(cut first)* are the ones to drop if you are running behind; everything else is
load-bearing for the labwork. **The times below already include showing and discussing
the figures** — each figure is produced by the code cell above it, so run the notebook
once before the session.


> If you are short of time, the two figures you should not skip are **2** (why the step
> size has a ceiling) and **3** (the zigzag) — the labwork asks you to reproduce both.


| § | Section | min |
|---|---|---|
| 1 | The idea  — *Figure 1* | 5 |
| 2 | How far can we safely step?  — *Figure 2* | 8 |
| 3 | The rate, and the zigzag  — *Figure 3* | 9 |
| 4 | Armijo backtracking  — *Figure 4* | 9 |
| 5 | Momentum: the square root  — *Figure 5* | 7 |
| 6 | Stopping, and recording  *(cut first)* | 4 |
| 7 | Today's labs | 2 |
| | **total** | **44** |
| | **core only** | **40** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

# One consistent look for every figure in the lecture.
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# If this import fails:   pip install matplotlib

---

## 1. The idea

$\nabla f(x)$ points in the direction of steepest *increase*. So go the other way:

$$\boxed{x_{k+1} = x_k - \alpha\,\nabla f(x_k)}$$

Two questions, and the whole day is in them:

1. **Which direction?** ($-\nabla f$ is one choice, not the only one) → `IDirectionRule`
2. **How far?** (the step length $\alpha$) → `ILineSearch`

Separating those two questions is not a stylistic choice. It is what makes gradient
descent, momentum, Newton and damped Newton the *same loop* with different parts.

### The generic loop

```
x = x0
repeat:
    g = objective.gradient(x)
    d = direction.direction(objective, x, g)     # which way
    a = line_search.step(objective, x, g, d)     # how far
    x = x + a * d
    event = StepEvent(...)                       # record what happened
    for obs in observers: obs.on_step(event)
    if stop.should_stop(event): break
```

That is the entire `DescentOptimizer`. Everything else this week is an object plugged
into it.

A direction $d$ is a **descent direction** if $g^\top d < 0$ — moving along it decreases
$f$ to first order. For $d = -g$ this is $-\|g\|^2 < 0$, always true. For Newton's
direction on day 4 it is true only when the Hessian is positive definite, which is
exactly why day 4 needs damping.

In [ ]:
# The level sets of f, the gradient at a point, and every direction that goes downhill.
f2  = lambda X1, X2: 0.5 * (X1 ** 2 + 3.0 * X2 ** 2)
g2  = lambda x: np.array([x[0], 3.0 * x[1]])

x1 = np.linspace(-3, 3, 400)
x2 = np.linspace(-2, 2, 400)
X1, X2 = np.meshgrid(x1, x2)
Z = f2(X1, X2)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

# --- left: the gradient is perpendicular to the level set, and points UPHILL.
ax[0].contour(X1, X2, Z, levels=np.linspace(0.2, 12, 12), colors="0.6", linewidths=0.8)
for pt in [(2.0, 1.0), (-2.2, 0.7), (1.0, -1.4), (-1.2, -1.1)]:
    x = np.array(pt, dtype=float)
    g = g2(x)
    gh = g / np.linalg.norm(g) * 0.75
    ax[0].arrow(*x, *gh, color="crimson", width=0.035, length_includes_head=True, zorder=5)
    ax[0].arrow(*x, *(-gh), color="tab:blue", width=0.035, length_includes_head=True, zorder=5)
    ax[0].plot(*x, "ko", ms=4, zorder=6)
ax[0].plot([], [], color="crimson", lw=3, label=r"$+\nabla f$  (uphill)")
ax[0].plot([], [], color="tab:blue", lw=3, label=r"$-\nabla f$  (downhill)")
ax[0].plot(0, 0, "*", color="gold", ms=18, mec="k", mew=0.8, zorder=7, label="minimum")
ax[0].legend(loc="upper left", fontsize=8, framealpha=0.95)
ax[0].set_title("The gradient is perpendicular to the level set")
ax[0].set_xlabel("$x_1$"); ax[0].set_ylabel("$x_2$"); ax[0].set_aspect("equal")

# --- right: -grad is ONE descent direction. The whole half-plane works.
x0 = np.array([1.8, 0.9])
g0 = g2(x0)
ax[1].contour(X1, X2, Z, levels=np.linspace(0.2, 12, 12), colors="0.6", linewidths=0.8)

# Shade the true half-plane {d : g.d < 0}, as a mask over the whole axes.
HALF = g0[0] * (X1 - x0[0]) + g0[1] * (X2 - x0[1])
ax[1].contourf(X1, X2, (HALF < 0).astype(float), levels=[0.5, 1.5],
               colors=["tab:blue"], alpha=0.13, zorder=1)
perp = np.array([-g0[1], g0[0]]) / np.linalg.norm(g0)
seg = x0[:, None] + perp[:, None] * np.array([-6.0, 6.0])
ax[1].plot(seg[0], seg[1], "k--", lw=1.0, zorder=2)

arrows = [(-g0, "tab:blue", r"$-\nabla f$", (-22, -10)),
          (np.array([-1.0, 0.2]), "tab:green", "another", (-30, 8)),
          (np.array([0.6, -1.0]), "tab:green", "and another", (30, -8)),
          (np.array([0.9, 0.5]), "crimson", "uphill!", (26, 10))]
for d, col, name, off in arrows:
    dh = np.asarray(d, float)
    dh = dh / np.linalg.norm(dh) * 0.62
    dot = float(g0 @ dh)
    ax[1].arrow(*x0, *dh, color=col, width=0.028, length_includes_head=True, zorder=5)
    ax[1].annotate(f"{name}\n$g^\\top d$ = {dot:+.2f}", xy=x0 + dh, xytext=off,
                   textcoords="offset points", fontsize=7.5, color=col,
                   ha="center", va="center", zorder=6,
                   bbox=dict(fc="white", ec="none", alpha=0.75, pad=0.8))
ax[1].plot(*x0, "ko", ms=5, zorder=7)
ax[1].set_title(r"Shaded: every descent direction ($g^\top d < 0$)")
ax[1].set_xlabel("$x_1$"); ax[1].set_ylabel("$x_2$"); ax[1].set_aspect("equal")
ax[1].set_xlim(-0.2, 3.0); ax[1].set_ylim(-0.7, 1.9)

plt.tight_layout()
plt.show()

**Figure 1 — what a descent direction is.**

*Left:* at every point the gradient (red) is perpendicular to the level curve through it
and points **uphill**; $-\nabla f$ (blue) points downhill. That perpendicularity is not a
coincidence — moving *along* a level curve does not change $f$, so the rate of change in
that direction is zero, which is exactly $g^\top d = 0$.

*Right:* the blue arrow $-g$ is only **one** of the directions that go downhill. Every
arrow landing in the shaded half-plane has $g^\top d < 0$ and is a legal descent
direction; the red one has $g^\top d > 0$ and goes uphill. The dashed line is the
boundary, $g^\top d = 0$.

> **Why this matters for your code.** `IDirectionRule` exists precisely because the shaded
> region has more than one member. On day 4 Newton will pick a *different* arrow from that
> same region — and will need damping exactly when its arrow escapes the shaded half.

---

## 2. How far can we safely step?

Call $f$ **$L$-smooth** if its gradient is Lipschitz: $\|\nabla f(x) - \nabla f(y)\| \le L\|x-y\|$.
Equivalently, for a twice-differentiable $f$, $\nabla^2 f \preceq L I$ — curvature is
bounded above by $L$. Then for every $x$ and $y$:

$$f(y) \;\le\; f(x) + \nabla f(x)^\top (y - x) + \frac{L}{2}\|y - x\|^2$$

The quadratic on the right is an upper bound on $f$ that touches it at $x$. Minimize
*that* instead of $f$ — it is a parabola, so we can do it exactly. Substituting
$y = x - \alpha g$:

$$f(x - \alpha g) \;\le\; f(x) - \alpha\|g\|^2 + \frac{L\alpha^2}{2}\|g\|^2
= f(x) - \alpha\left(1 - \frac{L\alpha}{2}\right)\|g\|^2$$

### The descent lemma

The bracket is positive whenever $\alpha < 2/L$, so the step is guaranteed to decrease
$f$. Choosing $\alpha = 1/L$ maximizes the guaranteed decrease:

$$\boxed{f(x - \tfrac{1}{L}g) \;\le\; f(x) - \frac{1}{2L}\|g\|^2}$$

So: **$\alpha \le 1/L$ is safe, $\alpha \ge 2/L$ diverges.** You will trigger the
divergence deliberately in Lab 3 — it is worth seeing once, because it is the single most
common reason a training run explodes.

The catch: $L$ is usually unknown. Two ways out, and we do both. Today: **estimate it by
trying** (Armijo). Day 4: **measure the curvature** (Newton).

In [ ]:
# The descent lemma, drawn. Along the ray x - alpha*g, compare the true function
# with the quadratic upper bound that the L-smoothness inequality guarantees.
L_true, mu_true = 10.0, 1.0
A2 = np.diag([mu_true, L_true])
fq = lambda x: 0.5 * x @ A2 @ x
# Start where the gradient points mostly along the FLAT direction, so the curvature
# actually felt along the ray is well below L and the bound's slack is visible.
x_s = np.array([1.0, 0.05])
g_s = A2 @ x_s
fx, gn2 = fq(x_s), float(g_s @ g_s)

alphas = np.linspace(0, 4.6 / L_true, 500)
phi = np.array([fq(x_s - a * g_s) for a in alphas])     # the truth along the ray
bound = fx - alphas * gn2 + 0.5 * L_true * alphas ** 2 * gn2   # the guaranteed upper bound
tangent = fx - alphas * gn2                                     # first order only
a_best = alphas[int(np.argmin(phi))]                            # best step for THIS f

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))

ax[0].fill_between(alphas, phi, bound, color="crimson", alpha=0.10, zorder=1)
ax[0].plot(alphas, phi, lw=2.4, color="tab:blue", label=r"true $f(x-\alpha g)$")
ax[0].plot(alphas, bound, lw=2.2, color="crimson",
           label=r"upper bound $f(x)-\alpha\|g\|^2+\frac{L}{2}\alpha^2\|g\|^2$")
ax[0].plot(alphas, tangent, lw=1.2, color="0.45", ls=":", label="tangent (first order)")
ax[0].axhline(fx, color="0.3", lw=1.0, ls="--")
ax[0].annotate("$f(x)$", xy=(4.5 / L_true, fx), fontsize=8, va="bottom",
               ha="right", color="0.3")

ytxt = fx * 1.12
for a, lab, col in [(1 / L_true, "$1/L$\nbest\nguaranteed", "darkgreen"),
                    (2 / L_true, "$2/L$\nguarantee\ngone", "crimson")]:
    ax[0].axvline(a, color=col, lw=1.2, ls="--")
    ax[0].annotate(lab, xy=(a, ytxt), fontsize=7.5, color=col, ha="center", va="top",
                   bbox=dict(fc="white", ec="none", alpha=0.8, pad=0.6))
ax[0].plot(1 / L_true, fx - gn2 / (2 * L_true), "o", color="darkgreen", ms=9, zorder=6)
ax[0].plot(a_best, phi.min(), "o", color="tab:blue", ms=9, zorder=6)
ax[0].annotate(f"best step for THIS $f$\n($\\alpha$ = {a_best:.2f}) — but we\n"
               "have no way to know it",
               xy=(a_best, phi.min()), xytext=(-6, 34), textcoords="offset points",
               fontsize=7.5, color="tab:blue", ha="center",
               arrowprops=dict(arrowstyle="->", color="tab:blue", lw=1.0))
ax[0].set_xlabel(r"step length $\alpha$"); ax[0].set_ylabel("value")
ax[0].set_title("The bound (red) lies above the truth (blue)")
ax[0].legend(fontsize=7.5, loc="lower left")
ax[0].set_ylim(0, fx * 1.18)

# --- right: what those step sizes actually do, over 60 iterations.
def run(alpha, n=60):
    x, hist = np.array([1.0, 1.0]), []
    for _ in range(n):
        hist.append(fq(x))
        x = x - alpha * (A2 @ x)
    return np.array(hist)

for mult, col, style in [(0.5, "tab:green", "-"), (1.0, "darkgreen", "-"),
                         (1.9, "tab:orange", "-"), (2.1, "crimson", "-")]:
    h = run(mult / L_true)
    ax[1].semilogy(np.maximum(h, 1e-300), style, color=col, lw=1.8,
                   label=f"$\\alpha = {mult}/L$" + ("  DIVERGES" if mult > 2 else ""))
ax[1].set_xlabel("iteration $k$"); ax[1].set_ylabel("$f(x_k)$  (log scale)")
ax[1].set_title("Below $2/L$ it falls; above $2/L$ it explodes")
ax[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

**Figure 2 — why $\alpha < 2/L$ and not a hair more.**

*Left:* the red parabola is the upper bound the $L$-smoothness inequality gives us. It
touches the true curve at $\alpha = 0$ and lies above it everywhere; the shaded gap
between them is the **slack in the bound**. We cannot minimize the blue curve — we do not
know it, we only ever get to evaluate it point by point — so we minimize the red one
instead. Its minimum is at $\alpha = 1/L$ (green dot): the **best decrease we can
guarantee knowing only $L$**. At $\alpha = 2/L$ the red parabola has climbed back to
$f(x)$ and the guarantee is gone.

Notice how conservative that is. The blue dot is the step that is actually best here, and
it is more than three times larger than $1/L$. The bound has to be safe for *every*
$L$-smooth function, so on any particular one it leaves value on the table — which is
precisely the gap Armijo (§4) goes looking for.

*Right:* the same four step sizes, actually run. Note the shape of each curve —
$\alpha = 1/L$ and $\alpha = 1.9/L$ both converge, but $1.9/L$ is *faster*, because the
bound is pessimistic for this particular $f$. And $2.1/L$ does not merely converge
slowly; it diverges geometrically, gaining a constant factor every step.

> **This is the plot you will produce yourself in Lab 3.** When a training run's loss goes
> to `nan`, this figure is almost always the reason: the step crossed $2/L$.

In [ ]:
# f(x) = 0.5 * x.T A x, with L = largest eigenvalue of A.
A = np.diag([1.0, 100.0])
L = np.linalg.eigvalsh(A).max()
f = lambda x: 0.5 * x @ A @ x
grad = lambda x: A @ x

for alpha, label in [(1.0 / L, "1/L      (safe)"),
                     (1.9 / L, "1.9/L    (still < 2/L)"),
                     (2.1 / L, "2.1/L    (past the cliff)")]:
    x = np.array([1.0, 1.0])
    for _ in range(60):
        x = x - alpha * grad(x)
    print(f"alpha = {label:22s} -> f after 60 steps = {f(x):12.4e}")

---

## 3. The rate, and the zigzag

Take the quadratic $f(x) = \tfrac12 x^\top A x$ with $A$ symmetric positive definite,
eigenvalues in $[\mu, L]$, and $\kappa = L/\mu$ its condition number — the quantity we met
yesterday. With the optimal fixed step $\alpha = \frac{2}{L+\mu}$:

$$\|x_{k+1} - x^\star\| \;\le\; \frac{\kappa - 1}{\kappa + 1}\,\|x_k - x^\star\|$$

This is **linear convergence**: the error is multiplied by a constant each step, so it
falls geometrically and a log-scale plot of $\|\nabla f\|$ is a straight line. Read the
constant:

| $\kappa$ | rate | steps for 6 digits |
|---|---|---|
| 1 | 0 | 1 |
| 10 | 0.818 | ~70 |
| 100 | 0.980 | ~690 |
| $10^4$ | 0.9998 | ~69 000 |

**The number of iterations is proportional to $\kappa$.** Which is why yesterday's
standardization was not cosmetic.

### Why it zigzags

On an elongated bowl the negative gradient does *not* point at the minimum. It points
across the valley, because the steep direction dominates. So the iterate bounces between
the walls, making rapid progress on the narrow axis and almost none along the floor —
which is the direction it actually needs to travel.

> **Where $\frac{\kappa-1}{\kappa+1}$ comes from** *(worth two minutes — it is three lines)*
>
> For $f(x)=\frac12 x^\top A x$ the minimizer is $x^\star = 0$ and one step is
> $x_{k+1} = x_k - \alpha A x_k = (I - \alpha A)\,x_k$. So the error is multiplied by the
> matrix $I - \alpha A$, whose eigenvalues are $1 - \alpha\lambda_i$. The worst
> contraction factor is therefore
> $\max_i |1 - \alpha\lambda_i| = \max\big(|1-\alpha\mu|,\ |1-\alpha L|\big)$.
> That maximum is smallest when the two terms are equal — when the smallest and largest
> eigenvalue are mishandled *equally badly* — which happens at
> $\alpha = \frac{2}{L+\mu}$, and gives
> $1 - \frac{2\mu}{L+\mu} = \frac{L-\mu}{L+\mu} = \frac{\kappa-1}{\kappa+1}$.
>
> Read that again and the zigzag is no longer mysterious: the best fixed step is a
> *compromise* between a direction that needs a small step and one that needs a large
> one. It is too big for the steep direction (so that coordinate overshoots and flips
> sign every step) and far too small for the flat one (so that coordinate crawls).


In [ ]:
def gradient_descent(A, x0, alpha, n_steps):
    """Return the whole trajectory, so we can look at its shape."""
    xs = [np.asarray(x0, dtype=float)]
    for _ in range(n_steps):
        xs.append(xs[-1] - alpha * (A @ xs[-1]))
    return np.array(xs)

A = np.diag([1.0, 100.0])          # kappa = 100
mu, L = 1.0, 100.0
alpha_opt = 2.0 / (L + mu)

traj = gradient_descent(A, [1.0, 1.0], alpha_opt, 8)
print("first iterates (watch x2 flip sign every step -- that is the zigzag):")
for k, x in enumerate(traj):
    print(f"  k={k}  x = [{x[0]: .6f}, {x[1]: .6f}]")

In [ ]:
# And the predicted rate is the observed rate.
kappa = L / mu
predicted = (kappa - 1) / (kappa + 1)

traj = gradient_descent(A, [1.0, 1.0], alpha_opt, 1200)
errs = np.linalg.norm(traj, axis=1) / np.linalg.norm(traj[0])   # relative error, x* = 0
observed = (errs[400] / errs[300]) ** (1 / 100)                 # well into the asymptotic regime

print(f"predicted rate (kappa-1)/(kappa+1) = {predicted:.6f}")
print(f"observed rate                      = {observed:.6f}")

below = np.flatnonzero(errs < 1e-6)
print(f"\npredicted steps for 1e-6 : {int(np.ceil(np.log(1e-6) / np.log(predicted)))}")
print(f"measured steps for 1e-6  : {below[0]}")

In [ ]:
# The zigzag, and what the condition number does to the convergence rate.
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))

# --- left: the path itself. kappa = 10 (not 100) so the zigzag is visible on paper.
A_show = np.diag([1.0, 10.0])
mu_s, L_s = 1.0, 10.0
a_s = 2.0 / (L_s + mu_s)
path = gradient_descent(A_show, [4.0, 1.4], a_s, 14)

gx = np.linspace(-1.0, 5.0, 400)
gy = np.linspace(-1.6, 1.6, 400)
GX, GY = np.meshgrid(gx, gy)
GZ = 0.5 * (GX ** 2 + 10.0 * GY ** 2)
ax[0].contour(GX, GY, GZ, levels=np.geomspace(0.05, 14, 14), colors="0.65", linewidths=0.8)
ax[0].plot(path[:, 0], path[:, 1], "o-", color="crimson", ms=4, lw=1.4, zorder=5,
           label="gradient descent")
ax[0].plot(0, 0, "*", color="gold", ms=18, mec="k", mew=0.8, zorder=6, label="minimum")
ax[0].set_aspect("equal")
ax[0].set_title(f"The zigzag ($\\kappa = {L_s/mu_s:.0f}$, optimal fixed step)")
ax[0].set_xlabel("$x_1$  (flat direction)"); ax[0].set_ylabel("$x_2$  (steep direction)")
ax[0].legend(fontsize=8, loc="upper right")

# --- right: the rate is a straight line on a log scale, with slope set by kappa.
for k, col in [(1.0, "tab:green"), (10.0, "tab:blue"), (100.0, "tab:orange"),
               (1000.0, "crimson")]:
    Ak = np.diag([1.0, k])
    ak = 2.0 / (k + 1.0)
    tk = gradient_descent(Ak, [1.0, 1.0], ak, 800)
    ek = np.linalg.norm(tk, axis=1) / np.linalg.norm(tk[0])
    ax[1].semilogy(np.maximum(ek, 1e-18), color=col, lw=1.8,
                   label=f"$\\kappa$ = {k:.0f}   rate {(k-1)/(k+1):.4f}")
    cross = np.flatnonzero(ek < 1e-6)
    if cross.size:
        ax[1].plot(cross[0], 1e-6, "o", color=col, ms=7, mec="k", mew=0.6, zorder=6)
        ax[1].annotate(f"{cross[0]}", xy=(cross[0], 1e-6), xytext=(0, -15),
                       textcoords="offset points", fontsize=8, color=col, ha="center")
ax[1].axhline(1e-6, color="0.3", ls="--", lw=1.0)
ax[1].annotate("tolerance $10^{-6}$", xy=(790, 1.6e-6), fontsize=8, color="0.3", ha="right")
ax[1].set_xlabel("iteration $k$"); ax[1].set_ylabel("relative error  (log scale)")
ax[1].set_title("Straight lines = linear convergence; $\\kappa$ sets the slope")
ax[1].set_ylim(1e-13, 3)
ax[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

**Figure 3 — the zigzag, and the price of a bad condition number.**

*Left:* the actual path, on a bowl with $\kappa = 10$. Every step is perpendicular to the
level curve it starts on (Figure 1), and because the bowl is elongated that perpendicular
points **across** the valley rather than **along** it. The iterate crosses the floor of
the valley over and over, and the sign of $x_2$ flips every step — that is what the number
column in the previous cell was showing you. Progress along $x_1$, the direction it
actually has to travel, is what is left over.

*Right:* the same run for four condition numbers. Each is a straight line on a log scale —
that is what *linear convergence* means, a constant factor per step — and $\kappa$ sets
the slope. The marked dots are where each curve reaches the $10^{-6}$ tolerance, and they
reproduce the table above: $\kappa = 1$ in a single step, $\kappa = 10$ in about 70,
$\kappa = 100$ in about 690. **$\kappa = 1000$ never reaches the line at all** — the plot
runs out at 800 iterations and it needs roughly 6 900.

> $\kappa = 1$ converges in **one** step: the bowl is a perfect circle, so $-\nabla f$
> points straight at the minimum. Every method this week is, in one way or another, an
> attempt to make the problem *look* like that circle.

---

## 4. Armijo backtracking: finding $\alpha$ without knowing $L$

We cannot compute $L$, but we can *ask the function*. Start optimistic, and shrink until
the step is good enough.

What is "good enough"? Not merely $f(x + \alpha d) < f(x)$ — a sequence of decreasing
steps can decrease forever and converge nowhere. We demand decrease **proportional to
what the slope promised**. The directional derivative at $x$ along $d$ is $g^\top d < 0$,
so to first order $f(x+\alpha d) \approx f(x) + \alpha g^\top d$. The **Armijo condition**
asks for a fixed fraction $c_1$ of that predicted gain:

$$\boxed{f(x + \alpha d) \;\le\; f(x) + c_1\,\alpha\,g^\top d}, \qquad c_1 \in (0, 1)$$

Typically $c_1 = 10^{-4}$ — a very weak demand: "make at least $0.01\%$ of the progress
the slope predicted". It rules out steps that are far too long, and nothing else. That is
all we need.

**Backtracking.** Try $\alpha_0$ (usually $1$). If the condition fails, $\alpha \leftarrow \rho\alpha$
with $\rho = 0.5$, and try again. Give up after a budget and raise `LineSearchFailed` —
the optimizer catches it and reports `converged=False`. It must not crash.

### By hand

$f(x) = x^2$ at $x = 1$. Then $g = 2$, $d = -2$, $f(1) = 1$, $g^\top d = -4$.

- $\alpha = 1$: $x + \alpha d = -1$, $f = 1$. Need $1 \le 1 + 10^{-4}(1)(-4) = 0.9996$.
  **Rejected** — it jumped clean across the valley to the same height.
- $\alpha = 0.5$: $x + \alpha d = 0$, $f = 0$. Need $0 \le 1 + 10^{-4}(0.5)(-4) = 0.9998$.
  **Accepted** — and it landed exactly on the minimum.

In [ ]:
def armijo(f, x, g, d, alpha0=1.0, c1=1e-4, rho=0.5, max_backtracks=50, verbose=False):
    fx, slope = f(x), g @ d
    if slope >= 0:
        raise ValueError("not a descent direction")
    alpha = alpha0
    for _ in range(max_backtracks):
        ok = f(x + alpha * d) <= fx + c1 * alpha * slope
        if verbose:
            print(f"  alpha={alpha:7.4f}  f(x+ad)={f(x + alpha * d):9.6f}  "
                  f"threshold={fx + c1 * alpha * slope:9.6f}  -> {'ACCEPT' if ok else 'reject'}")
        if ok:
            return alpha
        alpha *= rho
    raise RuntimeError("LineSearchFailed")


f1 = lambda x: float(x @ x)
x, g = np.array([1.0]), np.array([2.0])
print("f(x) = x^2 at x = 1, direction d = -g:")
armijo(f1, x, g, -g, verbose=True)

In [ ]:
# What backtracking is actually looking at: one function of one variable.
A4 = np.diag([1.0, 4.0])
f4 = lambda x: 0.5 * x @ A4 @ x
x4 = np.array([1.0, 1.0])
g4 = A4 @ x4
d4 = -g4
fx4, slope4 = f4(x4), float(g4 @ d4)

aa = np.linspace(0, 1.0, 400)
phi4 = np.array([f4(x4 + a * d4) for a in aa])

fig, ax = plt.subplots(figsize=(7.6, 4.6))
ax.plot(aa, phi4, lw=2.4, color="tab:blue", label=r"$\varphi(\alpha) = f(x + \alpha d)$")
ax.plot(aa, fx4 + aa * slope4, ls=":", lw=1.4, color="0.45",
        label=r"tangent: $f(x) + \alpha g^\top d$")

for c1, col, ls in [(0.3, "crimson", "--"), (1e-4, "darkgreen", "-.")]:
    ax.plot(aa, fx4 + c1 * aa * slope4, ls=ls, lw=1.8, color=col,
            label=rf"Armijo line, $c_1 = {c1:g}$")

# Backtracking trace, using the exaggerated c1 so the rejections are visible.
c1_show, alpha = 0.3, 1.0
offsets = [(16, 6), (18, 10), (14, -26)]
for j in range(4):
    val, thr = f4(x4 + alpha * d4), fx4 + c1_show * alpha * slope4
    ok = val <= thr
    ax.plot(alpha, val, "o" if ok else "X", ms=12, zorder=7,
            color="darkgreen" if ok else "crimson", mec="k", mew=0.7)
    ax.annotate(f"$\\alpha$ = {alpha:g}\n{'ACCEPT' if ok else 'reject'}",
                xy=(alpha, val), xytext=offsets[min(j, 2)], textcoords="offset points",
                fontsize=8, ha="left", color="darkgreen" if ok else "crimson",
                bbox=dict(fc="white", ec="none", alpha=0.85, pad=1.2))
    if ok:
        break
    alpha *= 0.5

ax.set_xlabel(r"step length $\alpha$"); ax.set_ylabel(r"$\varphi(\alpha)$")
ax.set_title("Backtracking: halve $\\alpha$ until the curve dips under the Armijo line")
ax.set_xlim(-0.02, 1.12)
ax.set_ylim(-1.5, float(f4(x4 + 1.0 * d4)) * 1.15)
ax.legend(fontsize=8, loc="upper center")
plt.tight_layout()
plt.show()

print(f"f(x) = {fx4:.4f},  g.d = {slope4:.4f}")
print(f"with c1 = 1e-4 the first accepted alpha is "
      f"{next(a for a in [1.0, .5, .25, .125] if f4(x4 + a*d4) <= fx4 + 1e-4*a*slope4)}")
print(f"with c1 = 0.3  the first accepted alpha is "
      f"{next(a for a in [1.0, .5, .25, .125] if f4(x4 + a*d4) <= fx4 + 0.3*a*slope4)}")

**Figure 4 — the line search is a one-dimensional problem.**

Once the direction $d$ is chosen, $f$ restricted to the ray $x + \alpha d$ is a function
of the single variable $\alpha$ — the blue curve. That is all a line search ever looks at.

The **grey dotted** line is the tangent: what the slope *promised* we would gain. No step
can do better than it near $\alpha = 0$, and for larger $\alpha$ the curve bends away from
it. The Armijo condition accepts $\alpha$ when the blue curve is **below** a line of
reduced slope $c_1 g^\top d$:

- with the textbook $c_1 = 10^{-4}$ (green dash-dot) the line is almost flat — it is
  barely more demanding than "$f$ went down at all". It accepts $\alpha = 0.5$ here.
- with an exaggerated $c_1 = 0.3$ (red dashed, used for the markers so you can see the
  rejections) the demand is real: $\alpha = 1$ and $\alpha = 0.5$ are rejected and
  $\alpha = 0.25$ accepted.

**Why we still want the weak version.** The job of $c_1$ is *not* to find a good step — it
is to rule out catastrophically long ones, like the $\alpha = 1$ here that lands higher
than where we started. Being fussy costs function evaluations and buys very little,
because the next iteration will correct a mediocre step anyway.

> Note what backtracking never does: it never *increases* $\alpha$. Starting from
> $\alpha_0 = 1$ is therefore a real choice — it is the natural scale for Newton on day 4,
> where $\alpha = 1$ is the step the model asks for, and everything smaller is damping.

Now the payoff. Swap `FixedStep` for `Armijo` and **not one line of the loop changes** —
that is the open/closed principle with a measurable benefit attached.

In [ ]:
def descent(f, grad, x0, direction_rule, step_rule, n_steps):
    """The one loop. Note it knows nothing about WHICH direction or step rule."""
    x = np.asarray(x0, dtype=float)
    history = [np.linalg.norm(x)]                 # the minimizer is 0, so this is the error
    for _ in range(n_steps):
        g = grad(x)
        d = direction_rule(g)
        x = x + step_rule(f, x, g, d) * d
        history.append(np.linalg.norm(x))
    return x, np.array(history) / history[0]      # relative to the starting error


A = np.diag([1.0, 100.0])
f = lambda x: 0.5 * x @ A @ x
grad = lambda x: A @ x
steepest = lambda g: -g

_, h_fixed  = descent(f, grad, [1.0, 1.0], steepest,
                      lambda f, x, g, d: 1.0 / 100.0, 200)
_, h_armijo = descent(f, grad, [1.0, 1.0], steepest,
                      lambda f, x, g, d: armijo(f, x, g, d), 200)

print(f"fixed step 1/L : relative error after 200 steps = {h_fixed[-1]:.3e}")
print(f"armijo         : relative error after 200 steps = {h_armijo[-1]:.3e}")
print("\nSame loop, same direction rule. Only the injected ILineSearch changed.")

---

## 5. Momentum: the square root

Gradient descent forgets everything at each step. But the zigzag is *systematic*: the
oscillating components alternate sign and cancel, while the slow component along the
valley floor always points the same way and accumulates. So keep a running average of
past steps — the oscillations will cancel themselves and the drift will survive.

**Heavy ball** (Polyak):

$$d_{k} = -g_k + \beta\, d_{k-1}, \qquad x_{k+1} = x_k + \alpha d_k$$

The name is physical: a ball with inertia rolling down the valley, instead of a
memoryless particle. With tuned $\alpha$ and $\beta$ the rate becomes

$$\frac{\sqrt{\kappa} - 1}{\sqrt{\kappa} + 1} \qquad\text{instead of}\qquad \frac{\kappa - 1}{\kappa + 1}$$

A **square root**. For $\kappa = 10^4$ that is the difference between $10^4$ and $10^2$
iterations — two orders of magnitude for one extra vector of state.

> This is why `IDirectionRule` is an *object* and not a function: `HeavyBall` has to
> remember its previous direction. A stateless design could not express it.

In [ ]:
class HeavyBall:
    def __init__(self, beta):
        self.beta = beta
        self.prev = None

    def __call__(self, g):
        d = -g if self.prev is None else -g + self.beta * self.prev
        self.prev = d
        return d


A = np.diag([1.0, 100.0])
f = lambda x: 0.5 * x @ A @ x
grad = lambda x: A @ x
mu, L = 1.0, 100.0
kappa = L / mu

# Tuned heavy-ball parameters for a quadratic.
beta = ((np.sqrt(L) - np.sqrt(mu)) / (np.sqrt(L) + np.sqrt(mu))) ** 2
alpha_hb = (2 / (np.sqrt(L) + np.sqrt(mu))) ** 2

_, h_gd = descent(f, grad, [1.0, 1.0], lambda g: -g,
                  lambda f, x, g, d: 2.0 / (L + mu), 1500)
_, h_hb = descent(f, grad, [1.0, 1.0], HeavyBall(beta),
                  lambda f, x, g, d: alpha_hb, 1500)

def first_below(h, tol=1e-6):
    below = np.flatnonzero(h < tol)
    return int(below[0]) if below.size else None

print(f"kappa = {kappa:.0f}, stopping at relative error < 1e-6")
print(f"  gradient descent : {first_below(h_gd):5d} steps   "
      f"(rate {(kappa-1)/(kappa+1):.4f}, predicted {np.log(1e-6)/np.log((kappa-1)/(kappa+1)):.0f})")
print(f"  heavy ball       : {first_below(h_hb):5d} steps   "
      f"(rate {(np.sqrt(kappa)-1)/(np.sqrt(kappa)+1):.4f}, predicted "
      f"{np.log(1e-6)/np.log((np.sqrt(kappa)-1)/(np.sqrt(kappa)+1)):.0f})")
print(f"  Newton (day 4)   :     1 step   -- it solves the quadratic exactly")

In [ ]:
# Momentum: the same bowl, the same starting point, one extra vector of state.
class _HB:
    def __init__(self, beta):
        self.beta, self.prev = beta, None

    def __call__(self, g):
        d = -g if self.prev is None else -g + self.beta * self.prev
        self.prev = d
        return d


def path_of(A, x0, rule, alpha, n):
    x, xs = np.asarray(x0, float), [np.asarray(x0, float)]
    for _ in range(n):
        x = x + alpha * rule(A @ x)
        xs.append(x.copy())
    return np.array(xs)


fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.3))

# --- left: the two paths, kappa = 10 for legibility.
mu_v, L_v = 1.0, 10.0
A_v = np.diag([mu_v, L_v])
b_v = ((np.sqrt(L_v) - np.sqrt(mu_v)) / (np.sqrt(L_v) + np.sqrt(mu_v))) ** 2
a_v = (2 / (np.sqrt(L_v) + np.sqrt(mu_v))) ** 2

p_gd = path_of(A_v, [4.0, 1.4], lambda g: -g, 2.0 / (L_v + mu_v), 22)
p_hb = path_of(A_v, [4.0, 1.4], _HB(b_v), a_v, 22)

gx = np.linspace(-1.2, 5.0, 400); gy = np.linspace(-1.9, 1.9, 400)
GX, GY = np.meshgrid(gx, gy)
ax[0].contour(GX, GY, 0.5 * (GX ** 2 + L_v * GY ** 2),
              levels=np.geomspace(0.05, 14, 14), colors="0.72", linewidths=0.7)
ax[0].axhline(0, color="0.45", lw=1.0, ls="--")
ax[0].annotate("the valley floor", xy=(4.4, 0.06), fontsize=7.5, color="0.35", ha="right")

reached = {}
for p, col, mk, name in [(p_gd, "crimson", "o", "gradient descent"),
                         (p_hb, "tab:blue", "s", f"heavy ball ($\\beta$ = {b_v:.2f})")]:
    # Mark where each method has covered 7/8 of the distance along the valley.
    hit = np.flatnonzero(np.abs(p[:, 0]) < 0.5)
    k = int(hit[0]) if hit.size else None
    reached[name.split(" (")[0]] = k
    ax[0].plot(p[:, 0], p[:, 1], mk + "-", color=col, ms=3.5, lw=1.3, zorder=4,
               label=f"{name} — $|x_1|<0.5$ at step {k}")
    if k is not None:
        ax[0].plot(p[k, 0], p[k, 1], "o", ms=14, mfc="none", mec=col, mew=2.2, zorder=7)
ax[0].plot(0, 0, "*", color="gold", ms=18, mec="k", mew=0.8, zorder=8)
ax[0].set_aspect("equal"); ax[0].set_xlabel("$x_1$  (along the valley)")
ax[0].set_ylabel("$x_2$  (across it)")
ax[0].set_title("22 steps each, from the same start\n"
                "(circles: first time within 0.5 of the goal along $x_1$)", fontsize=9)
ax[0].legend(fontsize=7.5, loc="lower right")
ax[0].set_ylim(-2.3, 2.1)

# --- right: the rates, at kappa = 100.
ax[1].semilogy(np.maximum(h_gd, 1e-18), color="crimson", lw=1.8, label="gradient descent")
ax[1].semilogy(np.maximum(h_hb, 1e-18), color="tab:blue", lw=1.8, label="heavy ball")
kk = np.arange(0, 1200)
r_gd = (kappa - 1) / (kappa + 1)
r_hb = (np.sqrt(kappa) - 1) / (np.sqrt(kappa) + 1)
ax[1].semilogy(kk, r_gd ** kk, ls="--", color="crimson", lw=1.0,
               label=f"rate $(\\kappa-1)/(\\kappa+1)$ = {r_gd:.3f}")
ax[1].semilogy(kk, r_hb ** kk, ls="--", color="tab:blue", lw=1.0,
               label=f"rate $(\\sqrt{{\\kappa}}-1)/(\\sqrt{{\\kappa}}+1)$ = {r_hb:.3f}")
ax[1].axhline(1e-6, color="0.3", ls=":", lw=1.0)
ax[1].set_xlim(0, 900); ax[1].set_ylim(1e-9, 3)
ax[1].set_xlabel("iteration $k$"); ax[1].set_ylabel("relative error  (log scale)")
ax[1].set_title(f"$\\kappa$ = {kappa:.0f}: the square root, in steps")
ax[1].legend(fontsize=7.5)

plt.tight_layout()
plt.show()

**Figure 5 — what the memory buys.**

*Left:* both methods start at the same point and take the same number of steps. **Look at
$x_1$, not at how wiggly the paths are.** Heavy ball covers seven eighths of the valley in
6 steps; gradient descent needs 11. After all 22 steps gradient descent is still at
$|x| \approx 5\times10^{-2}$ while heavy ball is at $4\times10^{-5}$ — a thousandfold
difference, from one extra stored vector.

It is worth being precise about the mechanism, because the picture can mislead. Momentum
does **not** damp the oscillation across the valley — with these tuned parameters the blue
path actually swings *wider* than the red one. What it does is exploit the difference
between the two directions. Across the valley the gradient alternates sign every step, so
consecutive contributions to $d_k = -g_k + \beta d_{k-1}$ partly cancel. Along the floor
the gradient keeps the same sign, so those contributions **add**, and the running sum
approaches a geometric series: the effective step along the valley is multiplied by about
$\frac{1}{1-\beta}$. Here $\beta = 0.27$ gives $\approx 1.4$, and the larger tuned
$\alpha$ contributes another $1.3$ — together about $1.8\times$, which is exactly the
$11/6$ we just measured.

*Right:* the rates at $\kappa = 100$, with the two theoretical lines dashed. Gradient
descent tracks its bound almost exactly. Heavy ball does **not** sit on its dashed line —
it is slower than the bound early on, then parallel to it. That is the constant-factor
transient discussed below the previous cell: the bound describes the slope of the tail,
not the height of the curve.

> **Read the slopes, not the heights.** Two curves with the same slope on a log plot have
> the same *rate*; a vertical offset between them is a constant factor, which matters far
> less as the tolerance tightens.

Roughly **690 against 90 against 1**.

Note the honest gap: the asymptotic bound predicts $\approx 69$ steps for heavy ball and
we measured about $93$. The bound describes the *asymptotic* contraction, and in the
tuned regime the heavy-ball iteration matrix has complex eigenvalues — the iterate
spirals toward the minimum rather than sliding straight in, and it pays a constant-factor
transient before the asymptotic rate takes over. The headline survives intact: a
$\sqrt{\kappa}$ method needs roughly $7\times$ fewer iterations than a $\kappa$ method
here, and the advantage grows as $\kappa$ does.

This is worth noticing in itself. **A rate is a statement about the tail, not a promise
about iteration 10.** You will meet the same distinction on day 4, where Newton's
*quadratic* convergence only appears once the iterate is close enough.

Those three numbers are the story of the week:

- **gradient descent** uses the slope → cost $\propto \kappa$;
- **momentum** uses the slope plus a memory → cost $\propto \sqrt{\kappa}$;
- **Newton** uses the actual curvature → cost independent of $\kappa$ (day 4).

Curvature information is what you are paying for, in every case.

---

## 6. Stopping, and recording

### When to stop

The mathematical condition is $\nabla f(x) = 0$, so in practice:

- $\|\nabla f(x)\| \le \varepsilon$ — **genuine convergence**;
- iteration budget exhausted — **not** convergence;
- the line search failed — **not** convergence.

Only the first justifies `converged=True`. Be strict about this: an optimizer that
reports success on hitting `max_iter` will waste hours of somebody's life. `AnyOf`
composes several criteria so the loop still sees exactly one.

### Recording

The loop should not also be in the business of collecting statistics. It emits a
`StepEvent` per iteration and hands it to every `IObserver`; `History` stores them. The
loop does not know whether anyone is listening, and adding a second observer changes
nothing inside it. One responsibility per class — the SRP of Week 1, Unit 4, with a
concrete payoff: you can plot the convergence of *any* optimizer you write this week
without touching any of them.

---

## 7. Today's labs

| Lab | What | The point |
|---|---|---|
| 1 (55 min) | `DescentOptimizer`, `SteepestDescent`, `FixedStep`, stopping, `History` | write the loop **once** |
| 2 (55 min) | `Armijo` | injected in place of `FixedStep`, with **no edit to the loop** |
| 3 (50 min) | `HeavyBall`, conditioning study | recover the rates; force a divergence |

In Lab 3, set the step above $2/L$ and watch it explode. Then explain it with the descent
lemma. **An algorithm you have only seen succeed is one you do not yet understand.**

**Two questions for the debrief:**

1. Why is $\alpha \le 1/L$ safe — where exactly does the $2$ in $2/L$ come from?
2. Why does momentum help? Answer in terms of what happens to the oscillating components
   versus the drift along the valley.

> **Tomorrow.** Every gradient today cost a pass over all $n$ samples. When $n$ is
> $10^6$, that is intolerable — so we will use a *sample* of the gradient instead, and
> discover that being approximately right much more often beats being exactly right.